# E-commerce Furniture Listing Analysis
**Project 01 | Data cleaning, exploratory analysis, and business findings**

This notebook analyzes the supplied product-listing snapshot. It does not infer customer behaviour, seasonality, profit, or transaction revenue because those fields are not present in the source file.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_FILE = PROJECT_DIR / 'data' / 'raw' / 'ecommerce_furniture_dataset_2024.csv'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
PROCESSED_FILE = PROCESSED_DIR / 'cleaned_furniture_data.csv'
FIGURE_DIR = PROJECT_DIR / 'reports' / 'figures'
print('Project directory:', PROJECT_DIR)
print('Raw file exists:', RAW_FILE.exists())

## 2. Load and inspect the raw data

In [ ]:
df = pd.read_csv(RAW_FILE, encoding='utf-8-sig')
df.columns = df.columns.str.strip()
display(df.head())
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.info())
display(df.isna().sum().rename('missing_values').to_frame())

## 3. Clean and prepare fields

Original prices are retained as nullable values. Rows are removed only when product title, current price, or units sold cannot support the core analysis.

In [ ]:
required = {'productTitle', 'originalPrice', 'price', 'sold', 'tagText'}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f'Missing expected columns: {sorted(missing)}')

data = df.copy()
data['productTitle'] = data['productTitle'].astype('string').str.strip()

def parse_currency(series):
    values = series.astype('string').str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
    return pd.to_numeric(values, errors='coerce')

def parse_units(series):
    values = series.astype('string').str.replace('+', '', regex=False).str.replace(',', '', regex=False).str.strip()
    return pd.to_numeric(values, errors='coerce')

data['price'] = parse_currency(data['price'])
data['originalPrice'] = parse_currency(data['originalPrice'])
data['sold'] = parse_units(data['sold'])
data['tagText'] = data['tagText'].astype('string').str.strip()
data = data.dropna(subset=['productTitle', 'price', 'sold']).copy()
data = data[(data['price'] >= 0) & (data['sold'] >= 0)].copy()

data['shipping_group'] = np.select(
    [data['tagText'].eq('Free shipping'), data['tagText'].str.startswith('+Shipping:', na=False)],
    ['Free shipping', 'Paid shipping'],
    default='Missing or other'
)
eligible = data['originalPrice'].notna() & data['originalPrice'].gt(0) & data['price'].gt(0)
data['discount_percentage'] = np.nan
data.loc[eligible, 'discount_percentage'] = (data.loc[eligible, 'originalPrice'] - data.loc[eligible, 'price']) / data.loc[eligible, 'originalPrice'] * 100
data.loc[data['discount_percentage'] < 0, 'discount_percentage'] = np.nan
data['estimated_listed_value'] = data['price'] * data['sold']

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
data.to_csv(PROCESSED_FILE, index=False)
print('Cleaned shape:', data.shape)
print('Saved:', PROCESSED_FILE)

## 4. Data quality and summary metrics

In [ ]:
summary = pd.Series({
    'Rows after cleaning': len(data),
    'Distinct product titles': data['productTitle'].nunique(),
    'Duplicate title rows beyond first occurrence': len(data) - data['productTitle'].nunique(),
    'Reported units sold': data['sold'].sum(),
    'Median listed price': data['price'].median(),
    'Rows with valid discount percentage': data['discount_percentage'].notna().sum(),
})
display(summary.to_frame('value'))

## 5. Exploratory analysis

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(data['price'], bins=40, edgecolor='white')
ax.set(title='Distribution of Current Listed Prices', xlabel='Price', ylabel='Number of listings')
plt.tight_layout()
plt.show()

In [ ]:
top_products = (data.groupby('productTitle')['sold'].sum().sort_values(ascending=False).head(10).sort_values())
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(top_products.index.str.slice(0, 58), top_products.values)
ax.set(title='Top 10 Product Listings by Reported Units Sold', xlabel='Reported units sold', ylabel='')
plt.tight_layout()
plt.show()
display(top_products.sort_values(ascending=False).to_frame('reported_units_sold'))

In [ ]:
shipping_counts = data['shipping_group'].value_counts().reindex(['Free shipping', 'Paid shipping', 'Missing or other'], fill_value=0)
display(shipping_counts.to_frame('listing_count'))
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(shipping_counts.index, shipping_counts.values)
ax.set(title='Shipping Label Mix', xlabel='Shipping group', ylabel='Number of listings')
ax.tick_params(axis='x', rotation=12)
plt.tight_layout()
plt.show()

In [ ]:
discounts = data['discount_percentage'].dropna()
print('Discount-eligible listings:', len(discounts))
if not discounts.empty:
    display(discounts.describe().to_frame('discount_percentage'))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(discounts, bins=30, edgecolor='white')
    ax.set(title='Discount Percentage Distribution (Eligible Listings)', xlabel='Discount (%)', ylabel='Number of listings')
    plt.tight_layout()
    plt.show()

## 6. Export Tableau-ready data

The cleaned CSV contains numeric prices and units, a grouped shipping field, and discount percentage only where a valid original price exists. The `estimated_listed_value` field is a price-times-units proxy, not verified realized revenue.

In [ ]:
TABLEAU_FILE = PROCESSED_DIR / 'furniture_tableau_dataset.csv'
data.to_csv(TABLEAU_FILE, index=False)
print('Tableau-ready dataset saved:', TABLEAU_FILE)

## 7. Business interpretation and limitations

- Compare listings by reported units sold; repeated titles should be aggregated when discussing product-level totals.
- Discount analysis covers only rows with usable original and current prices.
- Shipping labels are grouped into free, paid, and missing/other to avoid treating every shipping amount as a separate category.
- The source has no dates, customer identifiers, product categories, costs, or transaction-level records. Do not infer seasonality, customer behaviour, profit, or actual revenue.
- Predictive modelling is deferred until a suitable labelled dataset and validation design are available.